# ニュースから出来事（誰が・いつ・どこで・何をした）を抜き出して時間順に並べる（イベント抽出）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_semantics/event_extraction.ipynb)

NLP クックブック 17-12 ／ 意味解析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

ニュース記事から「誰が・何を・どこで・いつ・どうした」の組を出来事ごとに取り出し、起きた順に並べたい。この章の部品（述語項構造、時間表現の正規化、否定・モダリティ）を組み合わせる方法と、小さなオープン LLM に JSON で抜き出させる方法を、手作りの小さな正解で比べる。

In [1]:
!pip install -q kwja ja-timex

In [2]:
import re
import pandas as pd

# 架空のニュース 5 本（このレシピのために作ったもの）。日付は記事の掲載日で、「19日」「昨年」「来月」を解釈する基準になる
DOCS = [("2026-11-20", "みなと市の山本市長は19日、市役所で記者会見を開き、新しい図書館が2028年に開館する予定だと発表した。市は昨年、駅前の土地を購入していた。"),
        ("2026-11-21", "さくら電機の大阪工場で19日夜、火災が起きた。けが人はいなかった。同社は20日、来月にも生産を再開する見通しだと明らかにした。"),
        ("2026-11-22", "東西大学の研究チームは、新しい電池を開発したと21日に発表した。チームは2030年までに実用化を目指す。ただ、量産の方法はまだ決まっていない。"),
        ("2026-11-23", "北川選手は22日、東京で行われたマラソン大会で優勝した。北川選手は昨年の大会では途中で棄権していた。来年はアメリカで練習するかもしれない。"),
        ("2026-11-24", "会社員の男が23日、詐欺の疑いで県警に逮捕された。男は先月、高齢の女性から現金200万円をだまし取った疑いがある。その翌日には別の女性にも電話をかけていたという。")]

# 正解: (述語を見つける正規表現, 誰が, 何を, どこで, いつ, 事実性)。「|」は別解、「-」は書かれていない、「*」は採点しない
# いつ: 記事の日付から計算して YYYY / YYYY-MM / YYYY-MM-DD にする。事実性: 事実 / 否定 / 未確定（予定・推量・目標）
GOLD = [[("開[きくいけ]", "山本市長", "記者会見", "市役所", "2026-11-19", "事実"),
         ("開館", "図書館", "-", "-", "2028", "未確定"),
         ("発表", "山本市長", "*", "市役所|-", "2026-11-19", "事実"),     # 「何を」は「〜と」の引用（内容）なので採点しない
         ("購入", "市|みなと市", "土地|駅前の土地", "駅前|-", "2025", "事実")],
        [("起き", "火災", "-", "大阪工場|工場", "2026-11-19", "事実"),
         ("い[なる]", "けが人", "-", "大阪工場|工場|-", "2026-11-19|-", "否定"),
         ("再開", "さくら電機|同社", "生産", "大阪工場|工場|-", "2026-12", "未確定"),
         ("明らか", "さくら電機|同社", "*", "-", "2026-11-20", "事実")],
        [("開発", "研究チーム|チーム", "電池|新しい電池", "-", "-", "事実"),
         ("発表", "研究チーム|チーム", "*", "-", "2026-11-21", "事実"),
         ("目指", "研究チーム|チーム", "実用化", "-", "2030", "未確定"),
         ("決ま", "方法|量産の方法", "-", "-", "-", "否定")],
        [("行わ", "マラソン大会|大会", "-", "東京", "2026-11-22", "事実"),
         ("優勝", "北川選手", "-", "マラソン大会|東京", "2026-11-22", "事実"),
         ("棄権", "北川選手", "-", "大会|昨年の大会|-", "2025", "事実"),
         ("練習", "北川選手", "-", "アメリカ", "2027", "未確定")],
        [("逮捕", "県警", "男|会社員の男", "-", "2026-11-23", "事実"),             # 受け身: 逮捕したのは県警、されたのは男
         ("だまし取", "男|会社員の男", "現金|現金200万円", "-", "2026-10", "未確定"),  # 「疑いがある」なので確定していない
         ("かけ", "男|会社員の男", "電話", "-", "2026-10|-", "未確定")]]           # 「その翌日」は日付にできない。「という」は伝聞
SLOTS = ["who", "what", "where", "when", "fact"]

def match(gold, value, slot):
    """正解の別解のどれかに合えば True。時は完全一致、それ以外は 2 文字以上なら部分一致も認める（「市長」と「山本市長」）"""
    if gold == "*":
        return None
    v = "" if value is None else re.sub(r"\s", "", str(value))
    v = "" if v in ("null", "None", "なし", "不明") else v
    for g in gold.split("|"):
        if g == "-":
            if not v:
                return True
        elif v == g or (slot != "when" and len(v) >= 2 and (v in g or g in v)):
            return True
    return False

def evaluate(pred_docs):
    """pred_docs: 記事ごとの出来事のリスト。出来事は {"pred", "who", "what", "where", "when", "fact"} の辞書。
    正解の出来事ごとに、述語が合う予測を 1 つ探して対応づけ、項ごとに採点する（見つからなければその項はすべて誤り）"""
    found, hits, n = 0, {s: 0 for s in SLOTS}, {s: 0 for s in SLOTS}
    for gold, preds in zip(GOLD, pred_docs):
        used = set()
        for pat, *values in gold:
            j = next((j for j, p in enumerate(preds) if j not in used and re.search(pat, p["pred"] or "")), None)
            if j is not None:
                used.add(j); found += 1
            for slot, g in zip(SLOTS, values):
                ok = match(g, preds[j][slot] if j is not None else None, slot) if g != "*" else None
                if ok is not None:
                    n[slot] += 1; hits[slot] += bool(ok and j is not None)
    n_gold, n_pred = sum(map(len, GOLD)), sum(map(len, pred_docs))
    return {"出来事の再現率": f"{found}/{n_gold}", "出来事の適合率": f"{found}/{n_pred}",
            **{f"{s} の一致率": round(hits[s] / n[s], 2) for s in SLOTS}}

print(sum(map(len, GOLD)), "個の出来事。例:", GOLD[1][2])

19 個の出来事。例: ('再開', 'さくら電機|同社', '生産', '大阪工場|工場|-', '2026-12', '未確定')


In [3]:
# 方法 1: 部品を組み合わせる。KWJA の述語項構造（ガ格 = 誰が、ヲ格 = 何を、デ格 = どこで、時間格 = いつ）と、
# 用言の素性（否定表現・時制・モダリティ）で事実性を決め、ja-timex で時間表現を記事の日付から正規化する
import pathlib, subprocess
import pendulum
from ja_timex import TimexParser
from rhoknp import Document

for i, (_, text) in enumerate(DOCS):
    pathlib.Path(f"doc{i}.txt").write_text(text + "\n")
args = [a for i in range(len(DOCS)) for a in ("--filename", f"doc{i}.txt")]   # 1 回の起動でまとめて解析する（モデルの読み込みは 1 回）
out = subprocess.run(["kwja", *args, "--tasks", "char,word", "--model-size", "base"], capture_output=True, text=True, check=True).stdout
sents = {}                                        # S-ID の末尾は「記事の番号-文の番号」
for block in re.findall(r"# S-ID:.*?\nEOS\n", out + "\n", flags=re.S):
    doc_no = int(block.split()[1].split("-")[-2])
    sents.setdefault(doc_no, []).append(block)
kdocs = [Document.from_knp("".join(sents[i])) for i in range(len(DOCS))]

def content(bp):
    """項の基本句を含む文節から、助詞や記号を除いた部分（「山本市長は」→「山本市長」）。文章の外を指す項 (不特定:物 など) は空"""
    return "".join(m.text for m in bp.phrase.morphemes if m.pos not in ("助詞", "特殊", "判定詞"))

def normalize_time(t, ref):
    """ja-timex の値を、記事の日付 ref を基準に YYYY / YYYY-MM / YYYY-MM-DD にする"""
    if t.type == "DATE":
        y, m, d = (t.value[:10].split("-") + ["XX", "XX"])[:3]
        y = str(ref.year) if y == "XXXX" else y
        if d != "XX":
            return f"{y}-{f'{ref.month:02d}' if m == 'XX' else m}-{d}"
        return f"{y}-{m}" if m != "XX" else (y if t.value[:4] != "XXXX" else None)
    if t.type == "DURATION" and t.mod in ("BEFORE", "AFTER", "NOW"):        # 「昨年」= 1 年前、「来月」= 1 か月後
        n, unit = int(t.value[1:-1] or 0), t.value[-1]
        n = -n if t.mod == "BEFORE" else (0 if t.mod == "NOW" else n)
        if unit == "Y":
            return str(ref.add(years=n).year)
        if unit == "M":
            return ref.add(months=n).format("YYYY-MM")
        if unit == "D":
            return ref.add(days=n).format("YYYY-MM-DD")
    return None

def first_time(text, ref):
    times = [normalize_time(t, ref) for t in TimexParser(reference=ref).parse(text)]
    return next((x for x in times if x), None)

def last_time(text, ref):
    times = [normalize_time(t, ref) for t in TimexParser(reference=ref).parse(text)]
    return next((x for x in reversed(times) if x), None)

def factuality(bp):
    if "否定表現" in bp.features:
        return "否定"
    if bp.features.get("時制") == "非過去" or any(k.startswith("モダリティ-") for k in bp.features):
        return "未確定"                      # 「開館する予定だ」「再開する見通し」「練習するかもしれない」
    return "事実"

def pipeline(doc, date):
    ref = pendulum.parse(date, tz="Asia/Tokyo")
    events = []
    for sent in doc.sentences:
        for k, bp in enumerate(sent.base_phrases):
            # 動詞の述語で、節の主辞になっているものを出来事とする（「新しい」のような形容詞や「予定だ」は除く）
            if bp.features.get("用言") != "動" or "節-主辞" not in bp.features or not bp.pas:
                continue
            arg = {case: a[0] for case, a in bp.pas.get_all_arguments(relax=False).items() if a}
            text_of = lambda case: content(arg[case].base_phrase) if case in arg and hasattr(arg[case], "base_phrase") else None
            where = text_of("デ")
            if where is None and "ニ" in arg and hasattr(arg["ニ"], "base_phrase"):     # ニ格は地名のときだけ場所とみなす
                ne = [n for n in sent.named_entities if n.category.value == "LOCATION" and arg["ニ"].base_phrase.head in n.morphemes]
                where = text_of("ニ") if ne else None
            # いつ: 時間格の文節を正規化する。時間格がなければ、同じ文で述語より前にある最後の時間表現を使う
            when = first_time(arg["時間"].base_phrase.phrase.text, ref) if "時間" in arg and hasattr(arg["時間"], "base_phrase") else None
            if when is None:
                when = last_time("".join(b.text for b in sent.base_phrases[:k]), ref)
            events.append({"pred": bp.text.rstrip("。、"), "who": text_of("ガ"), "what": text_of("ヲ"), "where": where,
                           "when": when, "fact": factuality(bp)})
    return events

pipe_events = [pipeline(doc, date) for doc, (date, _) in zip(kdocs, DOCS)]
for i, events in enumerate(pipe_events):
    for e in events:
        print(i, e)
print(evaluate(pipe_events))

0 {'pred': '開き', 'who': '山本市長', 'what': '記者会見', 'where': '市役所', 'when': '2026-11-19', 'fact': '事実'}
0 {'pred': '開館する', 'who': '図書館', 'what': None, 'where': None, 'when': '2028', 'fact': '未確定'}
0 {'pred': '発表した', 'who': '山本市長', 'what': None, 'where': None, 'when': '2026-11-19', 'fact': '事実'}
0 {'pred': '購入していた', 'who': '市', 'what': '土地', 'where': None, 'when': '2025', 'fact': '事実'}
1 {'pred': '起きた', 'who': '火災', 'what': None, 'where': '大阪工場', 'when': '2026-11-19', 'fact': '事実'}
1 {'pred': 'いなかった', 'who': 'けが人', 'what': None, 'where': None, 'when': None, 'fact': '否定'}
1 {'pred': '再開する', 'who': '同社', 'what': '生産', 'where': None, 'when': '2026-12', 'fact': '未確定'}
1 {'pred': '明らかにした', 'who': '同社', 'what': None, 'where': None, 'when': '2026-11-20', 'fact': '事実'}
2 {'pred': '開発したと', 'who': '研究チーム', 'what': '電池', 'where': None, 'when': None, 'fact': '事実'}
2 {'pred': '発表した', 'who': '研究チーム', 'what': None, 'where': None, 'when': '2026-11-21', 'fact': '事実'}
2 {'pred': '目指す', 'who': 'チーム', 'what': 

In [4]:
# 方法 2: 小さなオープン LLM に JSON で抜き出させる（Qwen3-4B-Instruct-2507、Apache 2.0）
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.disable_progress_bar()   # 重みの読み込みの進み具合のバーを出さない

name = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(name)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

INSTRUCTION = """次のニュース記事から出来事を抜き出し、JSON だけを出力してください。
- 出来事は動詞 1 つにつき 1 つとし、記事に出てくる順に並べる。
- 各出来事は {"predicate": 動詞（記事の表記のまま）, "who": 動作や変化の主体, "what": 動作の対象（「〜を」に当たるもの）, "where": 場所, "when": 時, "factuality": 事実性}。
- when は記事の日付を基準に YYYY、YYYY-MM、YYYY-MM-DD のどれかに直す。
- factuality は「事実」「否定」「未確定」（予定・推量・目標など、まだ起きていないこと）のどれか。
- 書かれていない項目は null。主語が省略されていれば、文脈から補う。
出力の形: {"events": [...]}"""
# 形式を示す例を 1 つ付ける（評価の記事とは別に作ったもの）
EXAMPLE = ("記事の日付: 2026-10-05\n記事: 青空鉄道は4日、新しい駅を来年3月に開業すると発表した。工事は先月始まった。",
           {"events": [{"predicate": "開業する", "who": "青空鉄道", "what": "駅", "where": None, "when": "2027-03", "factuality": "未確定"},
                       {"predicate": "発表した", "who": "青空鉄道", "what": None, "where": None, "when": "2026-10-04", "factuality": "事実"},
                       {"predicate": "始まった", "who": "工事", "what": None, "where": None, "when": "2026-09", "factuality": "事実"}]})

@torch.no_grad()
def llm_extract(date, text):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{EXAMPLE[0]}"},
                {"role": "assistant", "content": json.dumps(EXAMPLE[1], ensure_ascii=False)},
                {"role": "user", "content": f"{INSTRUCTION}\n\n記事の日付: {date}\n記事: {text}"}]
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=600, do_sample=False)
    raw = tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        events = json.JSONDecoder().raw_decode(raw[raw.find("{"):])[0]["events"]
    except (ValueError, KeyError):
        print("JSON として読めない:", raw[:200]); return []
    fact = lambda f: "否定" if "否定" in str(f) else ("事実" if "事実" in str(f) else "未確定")
    return [{"pred": e.get("predicate"), "who": e.get("who"), "what": e.get("what"), "where": e.get("where"),
             "when": e.get("when"), "fact": fact(e.get("factuality"))} for e in events]

llm_events = [llm_extract(date, text) for date, text in DOCS]
for i, events in enumerate(llm_events):
    for e in events:
        print(i, e)
print(f"GPU メモリの最大: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
pd.DataFrame({"部品の組み合わせ": evaluate(pipe_events), "LLM (JSON)": evaluate(llm_events)}).T

0 {'pred': '開き', 'who': 'みなと市の山本市長', 'what': None, 'where': '市役所', 'when': '2026-11-19', 'fact': '事実'}
0 {'pred': '開館する', 'who': '新しい図書館', 'what': None, 'where': None, 'when': '2028', 'fact': '未確定'}
0 {'pred': '購入していた', 'who': '市', 'what': '駅前の土地', 'where': '駅前', 'when': '2025', 'fact': '事実'}
1 {'pred': '起きた', 'who': '火災', 'what': None, 'where': 'さくら電機の大阪工場', 'when': '2026-11-19', 'fact': '事実'}
1 {'pred': '明らかにした', 'who': '同社', 'what': '来月にも生産を再開する見通し', 'where': None, 'when': '2026-11-20', 'fact': '事実'}
2 {'pred': '開発した', 'who': '東西大学の研究チーム', 'what': '新しい電池', 'where': None, 'when': '2026-11-21', 'fact': '事実'}
2 {'pred': '目指す', 'who': 'チーム', 'what': '実用化', 'where': None, 'when': '2030', 'fact': '未確定'}
2 {'pred': '決まっていない', 'who': '量産の方法', 'what': None, 'where': None, 'when': None, 'fact': '未確定'}
3 {'pred': '優勝した', 'who': '北川選手', 'what': None, 'where': '東京', 'when': '2026-11-22', 'fact': '事実'}
3 {'pred': '棄権していた', 'who': '北川選手', 'what': None, 'where': None, 'when': '2025-11-22', 'fact': 

,出来事の再現率,出来事の適合率,who の一致率,what の一致率,where の一致率,when の一致率,fact の一致率
部品の組み合わせ,19/19,19/20,0.95,0.94,0.89,0.95,0.89
LLM (JSON),14/19,14/14,0.68,0.56,0.74,0.53,0.58


In [5]:
# 時間順に並べる。いつが分かった出来事だけを日付の文字列で並べる（粒度が違うと、「2026-11」は「2026-11-19」より前に来る）
def timeline(pred_docs):
    rows = [(e["when"], i, e["pred"], e["who"], e["fact"]) for i, events in enumerate(pred_docs) for e in events if e["when"]]
    return [f"{w:10} [{i}] {who or '（不明）'}が {p}（{f}）" for w, i, p, who, f in sorted(rows, key=lambda r: str(r[0]))]

for title, events in [("部品の組み合わせ", pipe_events), ("LLM", llm_events)]:
    print(f"--- {title}"); print("\n".join(timeline(events)))

--- 部品の組み合わせ
2025       [0] 市が 購入していた（事実）
2025       [3] 北川選手が 棄権していた（事実）
2026-10    [4] 男が だまし取った（事実）
2026-10    [4] 疑いが ある（未確定）
2026-11-19 [0] 山本市長が 開き（事実）
2026-11-19 [0] 山本市長が 発表した（事実）
2026-11-19 [1] 火災が 起きた（事実）
2026-11-20 [1] 同社が 明らかにした（事実）
2026-11-21 [2] 研究チームが 発表した（事実）
2026-11-22 [3] マラソン大会が 行われた（事実）
2026-11-22 [3] 北川選手が 優勝した（事実）
2026-11-23 [4] 男が 逮捕された（事実）
2026-11-25 [4] 男が かけていたと（事実）
2026-12    [1] 同社が 再開する（未確定）
2027       [3] 北川選手が 練習するかもしれない（未確定）
2028       [0] 図書館が 開館する（未確定）
2030       [2] チームが 目指す（未確定）
--- LLM
2025       [0] 市が 購入していた（事実）
2025-11-22 [3] 北川選手が 棄権していた（事実）
2026-10-23 [4] 男が だまし取った（事実）
2026-11-19 [0] みなと市の山本市長が 開き（事実）
2026-11-19 [1] 火災が 起きた（事実）
2026-11-20 [1] 同社が 明らかにした（事実）
2026-11-21 [2] 東西大学の研究チームが 開発した（事実）
2026-11-22 [3] 北川選手が 優勝した（事実）
2026-11-23 [4] 男が 逮捕された（事実）
2026-11-24 [4] 男が 電話をかけた（事実）
2027       [3] 北川選手が 練習するかもしれない（未確定）
2028       [0] 新しい図書館が 開館する（未確定）
2030       [2] チームが 目指す（未確定）


- イベント抽出は、出来事を表す語（述語）と、その参加者・時・場所を組にして取り出す課題である。この章の「文から関係の三つ組を抽出する (REBEL)」が 2 つのエンティティの間の関係を取るのに対し、ここでは述語を中心に、誰が・何を・どこで・いつを 1 つの組にする。方法 1 は、この章の「日本語の述語項構造解析をする (KWJA)」の格（ガ格・ヲ格・デ格・時間格）を項に当て、「日本語の時間表現を抽出して正規化する (ja-timex)」で時を日付に直し、事実性は KWJA が用言に付ける素性（否定表現・時制・モダリティ）から決めている。「否定・モダリティ・時制を取り出す」の規則を、述語から節の終わりまでの文字列にかけてもよい。
- 手作りの 19 個の出来事では、部品の組み合わせは出来事を 19 個すべて見つけ（余分が 1 個）、項の一致率は 0.89〜0.95 だった。LLM (Qwen3-4B-Instruct-2507) は 14 個しか見つけず、項の一致率は 0.53〜0.74 だった。ただし、記事も正解も筆者が作った 19 個だけで、「誰が = ガ格」「どこで = デ格」という決め方も KWJA の出力に近い。整った短いニュース文は解析器に有利でもある。一般的な優劣の結論ではなく、どこで間違えるかを見るための比較である。
- LLM は出来事を 5 個落とした。「〜と発表した」「〜と明らかにした」の 3 か所では、引用の中と外の述語のどちらか一方しか出さず、ほかに「いなかった」「行われた」も落とした。また、日付を細かくしすぎた（「昨年の大会」を 2025-11-22、「先月」を 2026-10-23）。「開発した」に発表の日付を付けた。「その翌日」を 2026-11-24 にしたのは根拠がない。事実性も、「疑いがある」「〜という」を事実とし、「決まっていない」を未確定とした。
- 部品の組み合わせの誤りは、どれも部品の限界によるものである。受け身の「男が県警に逮捕された」では、KWJA の格は表層の格なので、ガ格の「男」が「誰が」になる（逮捕したのは県警）。受け身・使役では格を入れ替える必要がある。「その翌日」は ja-timex が記事の日付の翌日と解釈し、2026-11-25（記事より後）になって、時系列でも逮捕の後に並んでしまった。前の出来事を基準にする時間表現は、その出来事の日付から計算しなければならない。「だまし取った疑いがある」は、述語自身の素性だけを見るので事実になり、「ある」が余分な出来事として拾われた。デ格は手段や原因も表すので、「途中」「疑い」が場所になった。
- 時系列は、日付の文字列で並べただけである。粒度の違う日付（「2026-10」と「2026-10-23」）の前後や、日付のない出来事の前後（「開発した」は「発表した」より前）は決められない。出来事の間の前後関係を直接推定する課題は時間関係の認識と呼ばれ、英語の TimeBank や TempEval、日本語の BCCWJ-TimeBank などのデータがある。
- 実用では、両方を組み合わせるとよい。解析器で述語と格を漏れなく取り出し、受け身の扱いや、時の基準の解決、事実性の判定のような文脈が必要なところを LLM に確かめさせる。LLM の出力は、大規模言語モデルの章の「LLM の出力から JSON を取り出して検証する」のように、形式と値（日付の書式、事実性のラベル）を検証してから使う。
- 検証の GPU では、KWJA のセルが 1 分弱、LLM のセルがモデルの読み込みを含めて 1 分余り（初回のモデルのダウンロードは別）で、LLM の GPU メモリの最大は 8.2 GB だった。`kwja` を入れると transformers などのバージョンが入れ替わるので、すでに transformers を読み込んでいたらランタイムを再起動する。
- 記事は架空のもの。KWJA と ja-timex は MIT ライセンス、Qwen3-4B-Instruct-2507 は Apache 2.0。